# CAMeL-BERT Inference Pipeline - FIXED VERSION

**Fix**: Now processes FULL document in overlapping chunks (not just first 512 tokens)

**Key Change**: `infer_with_offsets()` now chunks the full text and combines predictions

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"[OK] Working directory: {os.getcwd()}")

In [ ]:
!pip install transformers torch tqdm -q
print("[OK] Dependencies installed")

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
model_path = Path('checkpoints/camelbert_binary_classification_final')

print(f"[INFO] Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()

if torch.cuda.is_available():
    model = model.cuda()

print(f"[OK] Model loaded and ready")

In [ ]:
corpus_file = Path('data/processed/kitab_uqala_reference_corpus.txt')

print(f"[INFO] Loading corpus from {corpus_file}...")
with open(corpus_file, encoding='utf-8') as f:
    full_text = f.read()

print(f"[OK] Corpus loaded")
print(f"  Size: {len(full_text):,} chars")
print(f"  Words: {len(full_text.split()):,}")

In [ ]:
# ============================================================================
# FIXED: Process FULL document in overlapping chunks
# ============================================================================

def infer_with_offsets(text: str, tokenizer, model, chunk_size: int = 512, overlap: int = 50) -> dict:
    """
    Run inference on FULL document in overlapping chunks.
    
    FIX: Now handles documents larger than 512 tokens by chunking.
    Chunks overlap to catch boundaries at chunk edges.
    """
    print(f"[INFO] Processing {len(text):,} chars in chunks of {chunk_size} with {overlap} overlap...\n")
    
    all_predictions = []
    all_probabilities = []
    all_offsets = []
    all_tokens = []
    
    chunk_num = 0
    pos = 0
    
    while pos < len(text):
        chunk_num += 1
        chunk_start = pos
        chunk_end = min(pos + chunk_size, len(text))
        chunk = text[chunk_start:chunk_end]
        
        if chunk_num % 100 == 0 or chunk_num == 1:
            print(f"  Chunk {chunk_num}: chars {chunk_start:,}-{chunk_end:,} ({len(chunk)} chars)")
        
        # Tokenize this chunk
        encoded = tokenizer(
            chunk,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_offsets_mapping=True,
            return_tensors='pt'
        )
        
        # Run inference
        with torch.no_grad():
            if torch.cuda.is_available():
                input_ids = encoded['input_ids'].cuda()
                attention_mask = encoded['attention_mask'].cuda()
                outputs = model(input_ids, attention_mask=attention_mask)
            else:
                outputs = model(**encoded)
            
            logits = outputs.logits[0]  # [seq_len, 2]
        
        # Get predictions and probabilities
        preds = np.argmax(logits.cpu().numpy(), axis=-1)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[:, 1]
        
        tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
        offsets = encoded['offset_mapping'][0].numpy()
        
        # CRITICAL FIX: Adjust offsets by chunk position to get document-level positions
        adjusted_offsets = []
        for token_start, token_end in offsets:
            # Convert local offset (within chunk) to global document position
            global_start = chunk_start + token_start
            global_end = chunk_start + token_end
            adjusted_offsets.append((global_start, global_end))
        
        # Append to global lists
        all_predictions.extend(preds)
        all_probabilities.extend(probs)
        all_offsets.extend(adjusted_offsets)
        all_tokens.extend(tokens)
        
        # Move to next chunk (with overlap)
        pos += (chunk_size - overlap)
        
        if pos >= len(text):
            break
    
    print(f"\n[OK] Processed {chunk_num} chunks")
    print(f"[OK] Total tokens: {len(all_predictions):,}")
    print(f"[OK] Boundary tokens: {sum(all_predictions):,}")
    
    return {
        'predictions': all_predictions,
        'probabilities': all_probabilities,
        'tokens': all_tokens,
        'offsets': all_offsets,
    }

print("[OK] Chunked inference function defined")

In [ ]:
# Run inference (now processes FULL document)
print(f"[INFO] Running inference on full corpus...")
print(f"  Text length: {len(full_text):,} chars")
print(f"  This will process all {len(full_text):,} characters in overlapping chunks\n")

inference_result = infer_with_offsets(full_text, tokenizer, model, chunk_size=512, overlap=50)

In [ ]:
# Export results
export_data = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus',
        'text_size_chars': len(full_text),
        'text_size_words': len(full_text.split()),
        'model': 'camelbert_binary_classification_final',
        'fix_applied': 'Full document chunking with overlap'
    },
    'inference_results': {
        'total_tokens': len(inference_result['predictions']),
        'boundary_tokens': sum(inference_result['predictions']),
        'predictions': inference_result['predictions'],
        'probabilities': inference_result['probabilities'],
        'tokens': inference_result['tokens'],
        'offsets': inference_result['offsets'],
    }
}

output_file = Path('results/camelbert_kitab_uqala_raw_inference.json')
output_file.parent.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Saving inference results to {output_file}...")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"[OK] File saved: {file_size_mb:.1f} MB")
print(f"[OK] Location: results/camelbert_kitab_uqala_raw_inference.json")

## Download Instructions

1. Refresh Files panel (left sidebar)
2. Navigate to `results/camelbert_kitab_uqala_raw_inference.json`
3. Right-click → Download
4. Save to local `results/` directory

Then run locally:
```bash
python3 scripts/camelbert_local_postprocess.py \
    --input results/camelbert_kitab_uqala_raw_inference.json \
    --output results/camelbert_kitab_uqala_segments.json
```